<a href="https://colab.research.google.com/github/5atish/TrainingCourse/blob/main/Hugging%20Face%20Agents%20Course%20/Unit%203.%20Use%20Case%20for%20Agentic%20RAG/Building_and_Integrating_Tools_for_Your_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 52.2 MB/s eta 0:00:00


In [10]:
from smolagents import DuckDuckGoSearchTool

# Initialize the DuckDuckGo search tool
search_tool = DuckDuckGoSearchTool()

# Example usage
results = search_tool("Who's the current President of France?")
print(results)

## Search Results

[President of France - Wikipedia](https://en.wikipedia.org/wiki/President_of_France)
1 day ago - The president of France is the ex officio co-prince of Andorra, grand master of the Legion of Honour and of the National Order of Merit, and protector of the Institut de France in Paris. The officeholder is also honorary proto-canon of the Archbasilica of Saint John Lateran in Rome, although some have rejected the title in the past. The current ...

[Emmanuel Macron - Wikipedia](https://en.wikipedia.org/wiki/Emmanuel_Macron)
3 weeks ago - Macron later went to Puy du Fou ... the current government. On 30 August 2016, Macron resigned from the government ahead of the 2017 presidential election, to devote himself to his En Marche movement. There had been rising tensions and several reports that he had wanted to leave the Valls government since early 2015. He initially planned to leave after the cancellation of his "Macron 2" law but decided to stay on temporarily after a meet

In [2]:
!pip install llama-index-tools-duckduckgo

In [9]:
from llama_index.tools.duckduckgo import DuckDuckGoSearchToolSpec
from llama_index.core.tools import FunctionTool

# Initialize the DuckDuckGo search tool
tool_spec = DuckDuckGoSearchToolSpec()

search_tool = FunctionTool.from_defaults(tool_spec.duckduckgo_full_search)
# Example usage
response = search_tool("Who's the current President of France?")
print(response.raw_output[-1]['body'])

RatelimitException: https://links.duckduckgo.com/d.js?q=Who%27s+the+current+President+of+France%3F&kl=wt-wt&l=wt-wt&p=&s=0&df=&vqd=4-27174262660097529791824081231544426738&bing_market=wt-WT&ex=-1 202 Ratelimit

In [11]:
from smolagents import CodeAgent, OpenAIServerModel, Tool
from google.colab import userdata
import requests
import os

os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

class SerperSearchTool(Tool):
    name = "web_search"
    description = "Searches the web using Google (via Serper) and returns the top result snippets."
    inputs = {
        "query": {
            "type": "string",
            "description": "The search query."
        }
    }
    output_type = "string"

    def forward(self, query: str):
        headers = {
            "X-API-KEY": os.environ["SERPER_API_KEY"],
            "Content-Type": "application/json"
        }
        response = requests.post(
            "https://google.serper.dev/search",
            headers=headers,
            json={"q": query}
        )
        results = response.json()
        if "organic" in results and results["organic"]:
            top = results["organic"][0]
            return f"{top.get('title', '')}: {top.get('snippet', 'No snippet available.')}"
        return "No results found."

search_tool = SerperSearchTool()

model = OpenAIServerModel(
    model_id="openai/gpt-oss-120b",
    api_base="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"]
)

agent = CodeAgent(tools=[search_tool], model=model)

response = agent.run("Who's the current President of France?")
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Who's the current President of France?                                                                          │
│                                                                                                                 │
╰─ OpenAIModel - openai/gpt-oss-120b ─────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
                                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out: None

[Step 1: Duration 1.79 seconds| Input tokens: 2,081 | Output tokens: 115]

...

[Step 3: Duration 0.57 seconds| Input tokens: 6,479 | Output tokens: 357]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 
'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "python", "arguments": result = 
web_search(query="President of France")\nprint(result)}'}}

[Step 4: Duration 8.71 seconds]

AgentGenerationError: Error in generating model output:
Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "python", "arguments": result = web_search(query="President of France")\nprint(result)}'}}

In [1]:
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.core.tools import FunctionTool
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from google.colab import userdata
import requests
import os

os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

def serper_search(query: str) -> str:
    """Search the web using Serper (Google Search API) and return the top result snippet."""
    headers = {
        "X-API-KEY": os.environ["SERPER_API_KEY"],
        "Content-Type": "application/json"
    }
    response = requests.post(
        "https://google.serper.dev/search",
        headers=headers,
        json={"q": query}
    )
    results = response.json()
    if "organic" in results and results["organic"]:
        top = results["organic"][0]
        return f"{top.get('title', '')}: {top.get('snippet', 'No snippet available.')}"
    return "No results found."

search_tool = FunctionTool.from_defaults(serper_search)

llm = HuggingFaceInferenceAPI(
    model_name="Qwen/Qwen2.5-Coder-32B-Instruct",
    provider="auto",
    token=os.environ["HF_TOKEN"]
)

agent = AgentWorkflow.from_tools_or_functions(
    [search_tool],
    llm=llm
)

response = await agent.run("Who's the current President of France?")
print(response)

ClientResponseError: 402, message='Payment Required', url='https://router.huggingface.co/nscale/v1/chat/completions'

In [2]:
!pip install llama-index-llms-groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 950.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.1
    Uninstalling transformers-5.15.1:
      Successfully uninstalled transformers-5.15.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is i

In [2]:
from smolagents import Tool
import random

class WeatherInfoTool(Tool):
    name = "weather_info"
    description = "Fetches dummy weather information for a given location."
    inputs = {
        "location": {
            "type": "string",
            "description": "The location to get weather information for."
        }
    }
    output_type = "string"

    def forward(self, location: str):
        # Dummy weather data
        weather_conditions = [
            {"condition": "Rainy", "temp_c": 15},
            {"condition": "Clear", "temp_c": 25},
            {"condition": "Windy", "temp_c": 20}
        ]
        # Randomly select a weather condition
        data = random.choice(weather_conditions)
        return f"Weather in {location}: {data['condition']}, {data['temp_c']}°C"

# Initialize the tool
weather_info_tool = WeatherInfoTool()

In [3]:
import random
from llama_index.core.tools import FunctionTool

def get_weather_info(location: str) -> str:
    """Fetches dummy weather information for a given location."""
    # Dummy weather data
    weather_conditions = [
        {"condition": "Rainy", "temp_c": 15},
        {"condition": "Clear", "temp_c": 25},
        {"condition": "Windy", "temp_c": 20}
    ]
    # Randomly select a weather condition
    data = random.choice(weather_conditions)
    return f"Weather in {location}: {data['condition']}, {data['temp_c']}°C"

# Initialize the tool
weather_info_tool = FunctionTool.from_defaults(get_weather_info)

In [4]:
from langchain_core.tools import Tool
import random

def get_weather_info(location: str) -> str:
    """Fetches dummy weather information for a given location."""
    # Dummy weather data
    weather_conditions = [
        {"condition": "Rainy", "temp_c": 15},
        {"condition": "Clear", "temp_c": 25},
        {"condition": "Windy", "temp_c": 20}
    ]
    # Randomly select a weather condition
    data = random.choice(weather_conditions)
    return f"Weather in {location}: {data['condition']}, {data['temp_c']}°C"

# Initialize the tool
weather_info_tool = Tool(
    name="get_weather_info",
    func=get_weather_info,
    description="Fetches dummy weather information for a given location."
)

In [5]:
from smolagents import Tool
from huggingface_hub import list_models

class HubStatsTool(Tool):
    name = "hub_stats"
    description = "Fetches the most downloaded model from a specific author on the Hugging Face Hub."
    inputs = {
        "author": {
            "type": "string",
            "description": "The username of the model author/organization to find models from."
        }
    }
    output_type = "string"

    def forward(self, author: str):
        try:
            # List models from the specified author, sorted by downloads
            models = list(list_models(author=author, sort="downloads", direction=-1, limit=1))

            if models:
                model = models[0]
                return f"The most downloaded model by {author} is {model.id} with {model.downloads:,} downloads."
            else:
                return f"No models found for author {author}."
        except Exception as e:
            return f"Error fetching models for {author}: {str(e)}"

# Initialize the tool
hub_stats_tool = HubStatsTool()

# Example usage
print(hub_stats_tool("facebook")) # Example: Get the most downloaded model by Facebook

The most downloaded model by facebook is facebook/opt-125m with 11,448,106 downloads.


In [6]:
import random
from llama_index.core.tools import FunctionTool
from huggingface_hub import list_models

def get_hub_stats(author: str) -> str:
    """Fetches the most downloaded model from a specific author on the Hugging Face Hub."""
    try:
        # List models from the specified author, sorted by downloads
        models = list(list_models(author=author, sort="downloads", direction=-1, limit=1))

        if models:
            model = models[0]
            return f"The most downloaded model by {author} is {model.id} with {model.downloads:,} downloads."
        else:
            return f"No models found for author {author}."
    except Exception as e:
        return f"Error fetching models for {author}: {str(e)}"

# Initialize the tool
hub_stats_tool = FunctionTool.from_defaults(get_hub_stats)

# Example usage
print(hub_stats_tool("facebook")) # Example: Get the most downloaded model by Facebook

The most downloaded model by facebook is facebook/opt-125m with 11,448,106 downloads.


In [7]:
from langchain_core.tools import Tool
from huggingface_hub import list_models

def get_hub_stats(author: str) -> str:
    """Fetches the most downloaded model from a specific author on the Hugging Face Hub."""
    try:
        # List models from the specified author, sorted by downloads
        models = list(list_models(author=author, sort="downloads", direction=-1, limit=1))

        if models:
            model = models[0]
            return f"The most downloaded model by {author} is {model.id} with {model.downloads:,} downloads."
        else:
            return f"No models found for author {author}."
    except Exception as e:
        return f"Error fetching models for {author}: {str(e)}"

# Initialize the tool
hub_stats_tool = Tool(
    name="get_hub_stats",
    func=get_hub_stats,
    description="Fetches the most downloaded model from a specific author on the Hugging Face Hub."
)

# Example usage
print(hub_stats_tool.invoke("facebook")) # Example: Get the most downloaded model by Facebook

The most downloaded model by facebook is facebook/opt-125m with 11,448,106 downloads.


In [16]:
from smolagents import CodeAgent, InferenceClientModel

# Initialize the Hugging Face model
model = InferenceClientModel()

# Create Alfred with all the tools
alfred = CodeAgent(
    tools=[search_tool, weather_info_tool, hub_stats_tool],
    model=model
)

# Example query Alfred might receive during the gala
response = alfred.run("What is Facebook and what's their most popular model?")

print("🎩 Alfred's Response:")
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is Facebook and what's their most popular model?                                                           │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Our latest automated health check on model 'Qwen/Qwen3-Next-80B-A3B-Thinking' for provider 'novita' did not complete successfully.  Inference call might fail.


Error in generating model output:
500 Server Error: Internal Server Error for url: https://router.huggingface.co/novita/v3/openai/chat/completions 
(Request ID: Root=1-6a947675-582f190972bbdfc53ac2ab9d;6c765355-534f-4773-b306-d84821218734)

[Step 1: Duration 0.47 seconds]

AgentGenerationError: Error in generating model output:
500 Server Error: Internal Server Error for url: https://router.huggingface.co/novita/v3/openai/chat/completions (Request ID: Root=1-6a947675-582f190972bbdfc53ac2ab9d;6c765355-534f-4773-b306-d84821218734)

In [19]:
from smolagents import CodeAgent, OpenAIServerModel
import os
from google.colab import userdata

model = OpenAIServerModel(
    model_id="openai/gpt-oss-120b",
    api_base="https://api.groq.com/openai/v1",
    api_key=userdata.get('GROQ_API_KEY') #os.environ["GROQ_API_KEY"]
)

alfred = CodeAgent(
    tools=[search_tool, weather_info_tool, hub_stats_tool],
    model=model
)

response = alfred.run("What is Facebook and what's their most popular model?")
print("🎩 Alfred's Response:")
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is Facebook and what's their most popular model?                                                           │
│                                                                                                                 │
╰─ OpenAIModel - openai/gpt-oss-120b ─────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Retrieve most downloaded model from Facebook on Hugging Face Hub                                               
  most_popular = hub_stats(author="facebook")                                                                      
  print("Hub stats result:", most_popular)                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Hub stats result: The most downloaded model by facebook is facebook/opt-125m with 11,448,106 downloads.

Out: None

[Step 1: Duration 1.84 seconds| Input tokens: 2,170 | Output tokens: 299]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Brief description of Facebook (the company)                                                                    
  facebook_description = (                                                                                         
      "Facebook is a global technology company best known for its social networking platform, "                    
      "which enables people to connect, share content, and communicate online. Founded in 2004 "                   
      "and now part of Meta Platforms, Inc., Facebook also offers services such as Messenger, "                    
      "Instagram, Oculus, and various tools for businesses and developers."                                        
  )                                                                                                                
                                                                                                                   
  # Information about the most downloaded model from Facebook on Hugging Face Hub                                  
  # The variable `most_popular` was obtained in a previous step.                                                   
  model_info = most_popular  # e.g., "The most downloaded model by facebook is facebook/opt-125m with 11,448,106   
  downloads."                                                                                                      
                                                                                                                   
  # Combine both pieces of information into a final answer                                                         
  final_text = f"{facebook_description}\n\nAccording to the Hugging Face Hub statistics, {model_info}."            
  final_answer(final_text)                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Facebook is a global technology company best known for its social networking platform, which enables 
people to connect, share content, and communicate online. Founded in 2004 and now part of Meta Platforms, Inc., 
Facebook also offers services such as Messenger, Instagram, Oculus, and various tools for businesses and 
developers.

According to the Hugging Face Hub statistics, The most downloaded model by facebook is facebook/opt-125m with 
11,448,106 downloads..

[Step 3: Duration 2.04 seconds| Input tokens: 7,291 | Output tokens: 937]

🎩 Alfred's Response:
Facebook is a global technology company best known for its social networking platform, which enables people to connect, share content, and communicate online. Founded in 2004 and now part of Meta Platforms, Inc., Facebook also offers services such as Messenger, Instagram, Oculus, and various tools for businesses and developers.

According to the Hugging Face Hub statistics, The most downloaded model by facebook is facebook/opt-125m with 11,448,106 downloads..


In [13]:
print(type(search_tool))
print(type(weather_info_tool))
print(type(hub_stats_tool))

<class '__main__.SerperSearchTool'>
<class '__main__.WeatherInfoTool'>
<class 'langchain_core.tools.simple.Tool'>


In [14]:
from smolagents import Tool
from huggingface_hub import list_models

class HubStatsTool(Tool):
    name = "hub_stats"
    description = "Fetches the most downloaded model from a specific author on the Hugging Face Hub."
    inputs = {
        "author": {
            "type": "string",
            "description": "The username of the model author/organization to find models from."
        }
    }
    output_type = "string"

    def forward(self, author: str):
        try:
            models = list(list_models(author=author, sort="downloads", limit=1))
            if models:
                model = models[0]
                return f"The most downloaded model by {author} is {model.id} with {model.downloads:,} downloads."
            else:
                return f"No models found for author {author}."
        except Exception as e:
            return f"Error fetching models for {author}: {str(e)}"

hub_stats_tool = HubStatsTool()

print(type(hub_stats_tool))  # should now print <class '__main__.HubStatsTool'>

<class '__main__.HubStatsTool'>


In [15]:
alfred = CodeAgent(
    tools=[search_tool, weather_info_tool, hub_stats_tool],
    model=model
)

In [11]:
from smolagents import Tool

class WeatherInfoTool(Tool):
    name = "weather_info"
    description = "Fetches weather information for a given location."
    inputs = {
        "location": {
            "type": "string",
            "description": "The location to get weather for."
        }
    }
    output_type = "string"

    def forward(self, location: str):
        # your actual weather-fetching logic here
        ...

weather_info_tool = WeatherInfoTool()

In [12]:
from smolagents import CodeAgent, InferenceClientModel, Tool
import requests
import os

class SerperSearchTool(Tool):
    name = "web_search"
    description = "Searches the web using Google (via Serper) and returns the top result snippets."
    inputs = {
        "query": {
            "type": "string",
            "description": "The search query."
        }
    }
    output_type = "string"

    def forward(self, query: str):
        headers = {
            "X-API-KEY": os.environ["SERPER_API_KEY"],
            "Content-Type": "application/json"
        }
        response = requests.post(
            "https://google.serper.dev/search",
            headers=headers,
            json={"q": query}
        )
        results = response.json()
        if "organic" in results and results["organic"]:
            top = results["organic"][0]
            return f"{top.get('title', '')}: {top.get('snippet', 'No snippet available.')}"
        return "No results found."

search_tool = SerperSearchTool()

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    provider="auto",
    token=os.environ["HF_TOKEN"]
)

alfred = CodeAgent(
    tools=[search_tool, weather_info_tool, hub_stats_tool],
    model=model
)

response = alfred.run("What is Facebook and what's their most popular model?")
print("🎩 Alfred's Response:")
print(response)

AssertionError: All elements must be instance of BaseTool (or a subclass)